<a href="https://colab.research.google.com/github/JoaoPauloRadd/Whisper-Qwen-gTTS/blob/main/Whisper_Qwen_gTTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Defino a linguagem que irei utilizar:

In [1]:
language=['pt','Brazilian Portuguese']

Com JavaScript, extraio o audio e passo para o python salvar e reproduzir.

In [31]:
# Referência: https://gist.github.com/korakot/c21c3476c024add656dc5f548b0bc9a2be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=10):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))
print('Pronto!\n')

Ouvindo...



<IPython.core.display.Javascript object>

Pronto!



Instalo o Whisper para transcrever o audio

In [2]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Processo de transcrição:

In [3]:
import whisper

model = whisper.load_model("small")
result = model.transcribe('/content/request_audio.wav', fp16=False, language=language[0])
transcription = result["text"]
print(transcription)

 Qual é a massa da terra?


Diferente do proposto no desafio, irei usar API do Hugging Faces em vez da API da OpenAi. Desta forma não terei dor de cabeça com chave de uso e limite de consumo da API. Usarei um modelo Qwen/Qwen2.5-0.5B-Instruct presente no hugging faces. Ele trará um processamento local e rápido dentro das limitações do notebook.

Qwen2.5 is the latest series of Qwen large language models. For Qwen2.5, we release a number of base language models and instruction-tuned language models ranging from 0.5 to 72 billion parameters.

Dependencias para o modelo:


In [4]:
!pip install -U transformers accelerate

  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.2-py3-none-any.whl (108 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.2 which is incompatible.


Requisição para o modelo processar a informação da fala transcrita:

In [5]:
from transformers import pipeline

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

pipe = pipeline(
    task="text-generation",
    model=model_id,
    dtype="auto",
    device_map="auto",
)

messages = [
    {
        "role": "system",
        "content": f"Answer only in {language[1]}. Be clear and concise."
    },
    {
        "role": "user",
        "content": transcription
    },
]

outputs = pipe(
    messages,
    max_new_tokens=96,   # menor = mais rápido
    do_sample=False,
)

response_qwen = outputs[0]["generated_text"][-1]["content"]
print(response_qwen)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A massa da Terra é de aproximadamente 5.93 x 10^24 quilogramas.


Importar o gtts para converter a resposar para voz:
(não importei pois parece ser já uma biblioteca local padrão para o colab)

Agora iremos sintetizar para áudio:

In [7]:
from gtts import gTTS

from IPython.display import Audio, display

gtts_object = gTTS(text=response_qwen, lang=language[0], slow=False)
response_audio = '/content/response_audio.wav'
gtts_object.save(response_audio)
display(Audio(response_audio, autoplay=True))